# Home Credit Default Risk: EDA inicial y baseline

Proyecto educativo basado exclusivamente en `application_train.csv` de la competición oficial [Home Credit Default Risk](https://www.kaggle.com/competitions/home-credit-default-risk). El archivo se obtiene tras acceder/aceptar las reglas de Kaggle; no se incorporan tablas auxiliares. El esquema comprobado (307.511 filas, 122 columnas, `SK_ID_CURR`, `TARGET`) coincide con el archivo oficial de entrenamiento.

**Objetivo:** estimar la clase `TARGET` (dificultad de pago) como ejercicio de clasificación. Este baseline no es apto para decisiones crediticias reales.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 30)
DATA_PATH = Path("data/raw/application_train.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"No se encontró {DATA_PATH}. Ejecuta el notebook desde la raíz del proyecto.")

## 1. Carga y comprobación del archivo

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Dimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas")
print("Columnas de identificación/objetivo:", [c for c in ["SK_ID_CURR", "TARGET"] if c in df])
assert df.shape == (307_511, 122), "El archivo no coincide con las dimensiones del train oficial esperado."
assert {0, 1} == set(df["TARGET"].dropna().unique()), "TARGET debe ser binario 0/1."
df.head()

## 2. Tipos, faltantes y duplicados

In [ ]:
print("\nTipos de datos:")
display(df.dtypes.value_counts().rename_axis("tipo").to_frame("columnas"))
missing = (df.isna().mean().mul(100).sort_values(ascending=False)
           .rename("porcentaje_faltante").to_frame())
display(missing.head(20).round(2))
print("Filas duplicadas exactas:", int(df.duplicated().sum()))
print("IDs duplicados:", int(df["SK_ID_CURR"].duplicated().sum()))

## 3. Distribuciones y variable objetivo

In [ ]:
target_counts = df["TARGET"].value_counts().sort_index()
target_share = df["TARGET"].value_counts(normalize=True).sort_index()
display(pd.DataFrame({"conteo": target_counts, "proporción": target_share}))
print(f"Proporción positiva (TARGET=1): {target_share.loc[1]:.4%}")
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="TARGET", ax=axes[0])
axes[0].set_title("Distribución de TARGET")
axes[0].set_xlabel("TARGET (1 = dificultad de pago)")
for col, ax in zip(["AMT_INCOME_TOTAL", "AMT_CREDIT"], axes[1:]):
    sns.histplot(data=df, x=col, bins=50, ax=ax)
    ax.set_title(f"Distribución de {col}")
plt.tight_layout()
plt.show()

## 4. Fuga de información y selección de variables

`SK_ID_CURR` es un identificador y se excluye de las features. `TARGET` solo se conserva como etiqueta. El archivo de entrenamiento contiene atributos declarados al solicitar el crédito y no incluye las tablas auxiliares de comportamiento posterior; por ello no se detecta una columna evidentemente derivada del resultado. Se revisaría cualquier variable con semántica posterior a la decisión antes de ampliar el proyecto. Los nombres de columnas y tipos se inspeccionan a continuación.

Las estadísticas EDA describen el archivo completo; para el modelo, el preprocesamiento se ajusta exclusivamente con el subconjunto train.

In [ ]:
print("Columnas (para revisión semántica):")
print(df.columns.tolist())
X = df.drop(columns=["SK_ID_CURR", "TARGET"])
y = df["TARGET"].astype("int8")
print(f"Features predictivas: {X.shape[1]}; TARGET se mantiene aparte: {y.name}")

## 5. Partición estratificada 60/20/20

In [ ]:
# Primero se reserva 40%; luego se divide esa reserva por igual en validación y test.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=42, stratify=y_tmp
)

split_summary = pd.DataFrame({
    "filas": [len(y_train), len(y_val), len(y_test)],
    "porcentaje": [len(y_train)/len(y), len(y_val)/len(y), len(y_test)/len(y)],
    "proporción_TARGET_1": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])
display(split_summary)

## 6. Pipeline y regresión logística baseline

In [ ]:
numeric_features = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number", "bool"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=200, solver="liblinear", random_state=42)),
])
print(f"Variables numéricas: {len(numeric_features)}; categóricas: {len(categorical_features)}")
model.fit(X_train, y_train)

## 7. Evaluación en validation

In [ ]:
val_probability = model.predict_proba(X_val)[:, 1]
val_prediction = (val_probability >= 0.5).astype("int8")
metrics = {
    "ROC-AUC": roc_auc_score(y_val, val_probability),
    "Average Precision": average_precision_score(y_val, val_probability),
    "Precision (umbral 0.5)": precision_score(y_val, val_prediction, zero_division=0),
    "Recall (umbral 0.5)": recall_score(y_val, val_prediction, zero_division=0),
}
print("Proporción positiva en validation:", f"{y_val.mean():.4%}")
display(pd.Series(metrics, name="validation"))
cm = confusion_matrix(y_val, val_prediction, labels=[0, 1])
display(pd.DataFrame(cm, index=["real_0", "real_1"], columns=["pred_0", "pred_1"]))

## Conclusiones iniciales

La clase positiva es minoritaria, por lo que Average Precision y Recall complementan ROC-AUC y precision. El umbral 0.5 se usa solo como referencia inicial. El conjunto test queda reservado para una evaluación final posterior y no se consulta para seleccionar ni ajustar el modelo. Este resultado es didáctico y no debe utilizarse para decisiones crediticias reales.

# Fase 2: comparación, tuning pequeño y selección

Se conservan exactamente los objetos `X_train`, `X_val`, `X_test` y sus etiquetas creados arriba. **`X_test` no se usa en ningún ajuste ni evaluación de esta fase.** Cada candidato ajusta su propio preprocesador solo con train. Se compara Average Precision como métrica principal por el desbalance; ROC-AUC es complementaria.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import (average_precision_score, f1_score,
                             precision_recall_curve, precision_score,
                             recall_score, roc_auc_score, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import time
import psutil

# Se incorpora el baseline ya entrenado en Fase 1, sin cambiar su configuración.
candidates = {
    "Logistic Regression (baseline)": (model, {
        "C": 1.0, "solver": "liblinear", "max_iter": 200, "class_weight": None
    }, model.predict_proba(X_val)[:, 1]),
}
fit_rows = []
process = psutil.Process()
peak_rss_mb = process.memory_info().rss / 1024**2

def record_candidate(name, estimator, params):
    global peak_rss_mb
    pipe = Pipeline([("preprocessor", clone(preprocessor)), ("classifier", estimator)])
    rss_before = process.memory_info().rss / 1024**2
    started = time.perf_counter()
    pipe.fit(X_train, y_train)
    elapsed = time.perf_counter() - started
    probability = pipe.predict_proba(X_val)[:, 1]
    rss_after = process.memory_info().rss / 1024**2
    peak_rss_mb = max(peak_rss_mb, rss_after)
    candidates[name] = (pipe, params, probability)
    fit_rows.append({"modelo": name, "ROC-AUC": roc_auc_score(y_val, probability),
                     "Average Precision": average_precision_score(y_val, probability),
                     "segundos_fit": elapsed, "RSS_post_fit_MiB": rss_after,
                     "RSS_delta_fit_MiB": rss_after-rss_before})

# Bosque pequeño y limitado para acotar memoria y tiempo.
rf_settings = [
    {"n_estimators": 100, "max_depth": 12, "min_samples_leaf": 50, "max_features": "sqrt"},
    {"n_estimators": 160, "max_depth": 16, "min_samples_leaf": 100, "max_features": "sqrt"},
]
for i, params in enumerate(rf_settings, start=1):
    record_candidate(f"Random Forest {i}",
        RandomForestClassifier(**params, n_jobs=2, random_state=42, class_weight=None), params)

xgb_settings = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.08, "min_child_weight": 10},
    {"n_estimators": 180, "max_depth": 4, "learning_rate": 0.06, "min_child_weight": 10},
]
for i, params in enumerate(xgb_settings, start=1):
    record_candidate(f"XGBoost {i}",
        XGBClassifier(**params, subsample=0.8, colsample_bytree=0.8,
                      reg_lambda=5, max_bin=128, tree_method="hist",
                      n_jobs=2, random_state=42, eval_metric="logloss",
                      objective="binary:logistic"), params)

comparison = pd.DataFrame(fit_rows).set_index("modelo")
baseline_scores = candidates["Logistic Regression (baseline)"][2]
baseline_row = pd.DataFrame([{"ROC-AUC": roc_auc_score(y_val, baseline_scores),
    "Average Precision": average_precision_score(y_val, baseline_scores),
    "segundos_fit": np.nan, "RSS_post_fit_MiB": np.nan, "RSS_delta_fit_MiB": np.nan}],
    index=["Logistic Regression (baseline)"])
comparison = pd.concat([baseline_row, comparison])
comparison["parámetros"] = [str(candidates[name][1]) for name in comparison.index]
display(comparison.sort_values("Average Precision", ascending=False).round(4))
print(f"RSS actual tras ajuste (MiB): {process.memory_info().rss/1024**2:.1f}; máximo observado al muestrear tras ajustes: {peak_rss_mb:.1f}")

## Curva Precision–Recall y umbrales

Se selecciona el candidato con mayor Average Precision en validation. Esta selección y la comparación de umbrales ocurren solo en validation; por ello las cifras son estimaciones de selección, no un resultado final independiente.

In [ ]:
ranked = comparison.sort_values("Average Precision", ascending=False)
best_name = ranked.index[0]
best_model, best_params, best_probability = candidates[best_name]
print("Mejor candidato por Average Precision:", best_name)
print("Parámetros de la configuración:", best_params)
precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_val, best_probability)
plt.figure(figsize=(7, 5))
plt.plot(recall_curve, precision_curve, label=f"{best_name} (AP={average_precision_score(y_val, best_probability):.3f})")
plt.axhline(y_val.mean(), linestyle="--", color="gray", label="Prevalencia positiva")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Precision–Recall en validation")
plt.legend(); plt.grid(alpha=0.25); plt.show()

# Umbrales predefinidos más un umbral que maximiza F1 en validation.
f1_values = 2 * precision_curve[:-1] * recall_curve[:-1] / np.maximum(precision_curve[:-1] + recall_curve[:-1], 1e-12)
best_f1_threshold = float(pr_thresholds[np.argmax(f1_values)])
thresholds = [0.10, 0.20, 0.30, 0.50, best_f1_threshold]
threshold_rows = []
for threshold in sorted(set(round(t, 6) for t in thresholds)):
    pred = (best_probability >= threshold).astype("int8")
    threshold_rows.append({
        "umbral": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "F1": f1_score(y_val, pred, zero_division=0),
        "matriz [ [TN, FP], [FN, TP] ]": confusion_matrix(y_val, pred, labels=[0,1]).tolist(),
    })
threshold_table = pd.DataFrame(threshold_rows).set_index("umbral")
display(threshold_table.round(4))
print(f"Umbral que maximiza F1 dentro de validation: {best_f1_threshold:.6f} (no es una estimación independiente).")

## Importancia de variables

Para el candidato ganador se muestran las importancias disponibles después de la codificación. Para XGBoost y Random Forest son importancias por reducción de impureza/ganancia, útiles como diagnóstico pero no causales. En variables categóricas, la importancia corresponde a cada categoría codificada.

In [ ]:
classifier = best_model.named_steps["classifier"]
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()
if hasattr(classifier, "feature_importances_"):
    importance = pd.Series(classifier.feature_importances_, index=feature_names)
    display(importance.sort_values(ascending=False).head(20).rename("importance").to_frame())
elif hasattr(classifier, "coef_"):
    importance = pd.Series(np.abs(classifier.coef_[0]), index=feature_names)
    display(importance.sort_values(ascending=False).head(20).rename("absolute_coefficient").to_frame())
else:
    print("Este estimador no expone importancia de variables directamente.")

print("No se ha consultado X_test ni y_test en la Fase 2.")

## Sensibilidad: variante sin `CODE_GENDER`

Como análisis acotado de sensibilidad, se vuelve a entrenar el candidato XGBoost 2 con los mismos hiperparámetros y exactamente el mismo train/validation, excluyendo `CODE_GENDER`. La métrica se compara en validation; no se consulta test ni se cambia el candidato automáticamente. La comparación de desempeño no constituye una evaluación de equidad.

In [ ]:
from sklearn.compose import ColumnTransformer

X_train_no_gender = X_train.drop(columns=["CODE_GENDER"])
X_val_no_gender = X_val.drop(columns=["CODE_GENDER"])
numeric_no_gender = X_train_no_gender.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_no_gender = X_train_no_gender.select_dtypes(exclude=["number", "bool"]).columns.tolist()
preprocessor_no_gender = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_no_gender),
    ("categorical", categorical_pipeline, categorical_no_gender),
])
xgb_no_gender = Pipeline([
    ("preprocessor", preprocessor_no_gender),
    ("classifier", XGBClassifier(
        n_estimators=180, max_depth=4, learning_rate=0.06, min_child_weight=10,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=5, max_bin=128,
        tree_method="hist", n_jobs=2, random_state=42, eval_metric="logloss",
        objective="binary:logistic")),
])
xgb_no_gender.fit(X_train_no_gender, y_train)
no_gender_probability = xgb_no_gender.predict_proba(X_val_no_gender)[:, 1]
gender_sensitivity = pd.DataFrame([
    {"variante": "XGBoost 2 con CODE_GENDER",
     "ROC-AUC": roc_auc_score(y_val, best_probability),
     "Average Precision": average_precision_score(y_val, best_probability)},
    {"variante": "XGBoost 2 sin CODE_GENDER",
     "ROC-AUC": roc_auc_score(y_val, no_gender_probability),
     "Average Precision": average_precision_score(y_val, no_gender_probability)},
]).set_index("variante")
gender_sensitivity.loc["cambio sin género - base"] = (gender_sensitivity.iloc[1] - gender_sensitivity.iloc[0]).values
display(gender_sensitivity.round(6))